# Credit Risk Assessment & Fraud Detection — End-to-End Walkthrough

This notebook walks through every phase of the project interactively.
For the full automated run, use `python src/pipeline.py` instead.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import data_generation, preprocessing as prep, feature_engineering as fe
import imbalance_handling as imb, credit_models as cm, fraud_models as fm
import ensemble as ens, evaluation as ev, interpretability as interp
import pandas as pd

## Phase 1: Data Understanding

In [ ]:
data_generation.main()
credit_df = pd.read_csv(ROOT / 'data/raw/credit_data.csv')
txn_df = pd.read_csv(ROOT / 'data/raw/transactions.csv')
credit_df.head()

In [ ]:
_ = prep.assess_data_quality(credit_df, 'credit_data')
_ = prep.assess_data_quality(txn_df, 'transactions')

## Phase 3: Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12,4))
credit_df['default'].value_counts().plot(kind='bar', ax=axes[0], title='Credit Default Distribution')
txn_df['is_fraud'].value_counts().plot(kind='bar', ax=axes[1], title='Fraud Distribution', color='orange')
plt.tight_layout(); plt.show()

In [ ]:
num_cols = credit_df.select_dtypes(include='number').columns
plt.figure(figsize=(10,8))
sns.heatmap(credit_df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Credit Feature Correlation'); plt.show()

## Phase 2 & 4: Preprocessing + Feature Engineering (Credit)

In [ ]:
df = prep.handle_missing_values(credit_df)
df = prep.treat_outliers(df, ['income','loan_amount','existing_debt'])
df = fe.engineer_credit_features(df)
df_enc, encoders = prep.encode_categoricals(df, ['home_ownership','loan_purpose'])
feature_cols = [c for c in df_enc.columns if c not in ('applicant_id','default')]
X, y = df_enc[feature_cols], df_enc['default']
X_train, X_test, y_train, y_test = prep.split_data(X, y)
num_c = X_train.select_dtypes(include='number').columns.tolist()
X_train, X_test, scaler = prep.scale_features(X_train, X_test, num_c)
X_train.head()

## Phase 5: Handling Imbalance

In [ ]:
comparison = imb.compare_resampling_techniques(X_train, y_train)
comparison

In [ ]:
X_train_sm, y_train_sm = imb.apply_smote(X_train, y_train)

## Phase 6-8: Models, Ensembles & Evaluation (Credit)

In [ ]:
models = cm.get_credit_models()
trained = cm.train_credit_models(models, X_train_sm, y_train_sm)
results = ev.compare_models(trained, X_test, y_test)
results

In [ ]:
best_name = results.iloc[0]['model']
best_model = trained[best_name]
ev.plot_confusion_matrix(best_model, X_test, y_test, best_name)
ev.plot_roc_curves(trained, X_test, y_test)
print('Best model:', best_name)

## Phase 9: Interpretability

In [ ]:
importance = interp.get_feature_importance(best_model, feature_cols)
interp.plot_feature_importance(importance)
importance.head(10)

In [ ]:
print(interp.extract_business_rules(X_train, y_train))

## Credit Risk Decision Support: PD to Expected Loss to Rating

In [ ]:
pd_scores = cm.predict_pd(best_model, X_test)
scorecard = cm.build_credit_scorecard(X_test, pd_scores, df.loc[X_test.index, 'loan_amount'].values)
scorecard.head(10)

## Fraud Detection Pipeline (repeat pattern)

Same steps apply to `transactions.csv` using `fraud_models.py`. See `src/pipeline.py` -> `run_fraud_pipeline()` for the full automated version, including anomaly detection and real-time scoring latency.